## 清华 EV Brand 2：四模型 × 三随机种子

运行车辆测试折 `fold=1–4`（fold=0 已完成）；主指标为车辆样本级 AP（Average Precision/PR-AUC），AUROC 为次指标。该 notebook 不执行测试标签训练。

In [1]:
import platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), '请在 Kaggle 设置中启用 GPU'
print('GPU:', torch.cuda.get_device_name(0))


Python: 3.12.13
PyTorch: 2.10.0+cu128
GPU: Tesla T4


In [2]:
%cd /kaggle/working
!rm -rf EnhancedMTADGAT
!git clone --depth 1 https://github.com/wonkawonka/EnhancedMTADGAT.git
%cd /kaggle/working/EnhancedMTADGAT
!git log -1 --oneline
!pip -q install -r requirements-kaggle-main.txt


/kaggle/working
Cloning into 'EnhancedMTADGAT'...
remote: Enumerating objects: 462, done.
remote: Counting objects: 100% (462/462), done.
remote: Compressing objects: 100% (261/261), done.
remote: Total 462 (delta 202), reused 387 (delta 190), pack-reused 0 (from 0)
Receiving objects: 100% (462/462), 99.00 MiB | 36.85 MiB/s, done.
Resolving deltas: 100% (202/202), done.
/kaggle/working/EnhancedMTADGAT
cb86cc2 (grafted, HEAD -> main, origin/main, origin/HEAD) fix: 稳定外部验证与统一AP报告


In [3]:
import os
from pathlib import Path

# Kaggle Input 的实际挂载路径；root 必须是三个品牌的共同父目录。
data_root = Path('/kaggle/input/datasets/daisychen2/tsinghua-ev2')
brand=2
# !find /kaggle/input -maxdepth 4 -type d | head -50
brand_root = data_root / f'battery_brand{brand}'
has_data = any((brand_root / folder).is_dir() for folder in ('data', 'train', 'test'))
if not (brand_root / 'label').is_dir() or not has_data:
    raise FileNotFoundError(f'清华 EV Brand{brand} 输入路径不可用：{brand_root}')
os.environ['MTAD_GAT_TSINGHUA_EV_ROOT'] = str(data_root)
os.environ['MTAD_GAT_DATASETS_ROOT'] = '/kaggle/working/EnhancedMTADGAT/datasets'
os.environ['MTAD_GAT_RUNS_ROOT'] = '/kaggle/working/EnhancedMTADGAT/runs'
print('清华 EV 数据根目录：', data_root)
print('品牌目录：', [p.name for p in data_root.iterdir() if p.is_dir()])


清华 EV 数据根目录： /kaggle/input/datasets/daisychen2/tsinghua-ev2
品牌目录： ['battery_brand2']


In [4]:
import json
from pathlib import Path

BRAND = 2
plan = {
  'plan_name': f'34_tsinghua_ev_brand{BRAND}_four_models_three_seeds_folds1to4',
  '_protocol': 'Brand 内车辆级 paper_protocol；运行 fold=1–4（fold=0 已完成），三个训练随机种子。训练与索引仅用训练/验证数据；测试标签仅用于最终 AP/AUROC。',
  '_metric': '主指标：车辆样本级 Average Precision (AP/PR-AUC)；次指标：AUROC；仅计五个响应通道。',
  'seeds': [3407, 2024, 2025],
  'common_args': {
    'epochs': 10, 'bs': 128, 'lookback': 127, 'battery_windows_per_snippet': 1,
    'battery_vehicle_top_ratio': 0.05, 'battery_split_protocol': 'paper_protocol', 'battery_fold_seed': 0,
    'battery_normalization': 'paper_channel', 'battery_response_only_training': False,
    'battery_score_channels': 'response', 'normalize': True, 'init_lr': 0.001, 'dropout': 0.3,
    'use_cuda': True, 'require_cuda': True, 'num_workers': 2, 'predict_num_workers': 2,
    'persistent_workers': True, 'early_stopping_patience': 3, 'early_stopping_min_delta': 0.0001,
    'log_tensorboard': False, 'dataset': 'TSINGHUA_EV', 'battery_brand': BRAND
  },
  'experiments': [
    {'name': 'c4_backbone', 'runner': 'src.runners.train_nc_battery', 'args': {
      'model_name': 'mtad_gat_c4_physics', 'use_transformer': True, 'use_regime_condition': True,
      'regime_encoder_type': 'temporal', 'regime_condition_mode': 'fusion', 'regime_aux_lambda': 0.05,
      'use_physical_state_encoding': True, 'use_physical_regularization': True,
      'use_physical_response_score': True, 'use_physical_consistency_head': False,
      'run_id': f'tsinghua_b{BRAND}_c4_backbone'}},
    {'name': 'c4_physical_consistency', 'runner': 'src.runners.train_nc_battery', 'args': {
      'model_name': 'mtad_gat_c4_physics', 'use_transformer': True, 'use_regime_condition': True,
      'regime_encoder_type': 'temporal', 'regime_condition_mode': 'fusion', 'regime_aux_lambda': 0.05,
      'use_physical_state_encoding': True, 'use_physical_regularization': True,
      'use_physical_response_score': True, 'use_physical_consistency_head': True,
      'physical_consistency_hidden_dim': 64, 'physical_consistency_latent_dim': 16,
      'physical_consistency_aux_weight': 1.0, 'physical_consistency_kl_weight': 0.0001,
      'physical_consistency_score_max_weight': 0.35, 'run_id': f'tsinghua_b{BRAND}_c4_physical_consistency'}}
  ]
}
for index,experiment in enumerate(plan['experiments']):
    if index==0:
        experiment['matrix']={'battery_fold': [2, 3, 4]}
    else:
        experiment['matrix'] = {'battery_fold': [1, 2, 3, 4]}
plan_path = Path(f'configs/internal/34_tsinghua_ev_brand{BRAND}_four_models_three_seeds_folds1to4.json')
plan_path.write_text(json.dumps(plan, ensure_ascii=False, indent=2), encoding='utf-8')
print(plan_path)
print('总训练数：', len(plan['experiments']) * len(plan['seeds']) * 4)


configs/internal/34_tsinghua_ev_brand2_four_models_three_seeds_folds1to4.json
总训练数： 24


In [5]:
# 先只展开计划，不训练；应显示 48 个实验。
!python run.py internal --plan configs/internal/34_tsinghua_ev_brand2_four_models_three_seeds_folds1to4.json --dry-run


/usr/bin/python3 -m src.runners.compare_experiments --plan configs/internal/34_tsinghua_ev_brand2_four_models_three_seeds_folds1to4.json --dry-run
Loaded 21 experiments from 34_tsinghua_ev_brand2_four_models_three_seeds_folds1to4.json

[1/21] c4_backbone_f2_seed3407
/usr/bin/python3 -m src.runners.train_nc_battery --epochs 10 --bs 128 --lookback 127 --battery_windows_per_snippet 1 --battery_vehicle_top_ratio 0.05 --battery_split_protocol paper_protocol --battery_fold_seed 0 --battery_normalization paper_channel --battery_response_only_training false --battery_score_channels response --normalize true --init_lr 0.001 --dropout 0.3 --use_cuda true --require_cuda true --num_workers 2 --predict_num_workers 2 --persistent_workers true --early_stopping_patience 3 --early_stopping_min_delta 0.0001 --log_tensorboard false --dataset TSINGHUA_EV --battery_brand 2 --model_name mtad_gat_c4_physics --use_transformer true --use_regime_condition true --regime_encoder_type temporal --regime_condition_m

In [6]:
# 正式训练。日志实时输出；中断后可通过 --resume --skip-existing 续跑。
!python run.py internal --plan configs/internal/34_tsinghua_ev_brand2_four_models_three_seeds_folds1to4.json --resume --skip-existing --only 


usage: run.py internal [-h] [--python PYTHON] [--dry-run] --plan PLAN
                       [--skip-existing] [--resume] [--only ONLY]
run.py internal: error: argument --only: expected one argument


In [7]:
import json, csv
from pathlib import Path

root = Path('runs/internal/34_tsinghua_ev_brand2_four_models_three_seeds_folds1to4/output')
rows = []
for path in sorted(root.glob('*/metrics.json')):
    result = json.loads(path.read_text(encoding='utf-8'))
    metrics = result['metrics']
    rows.append({
        'experiment': path.parent.name, 'brand': result['brand'], 'fold': result['fold'],
        'model': result['model_name'], 'AP': metrics['vehicle_average_precision'],
        'AUROC': metrics['vehicle_auroc'], 'F1': metrics['vehicle_f1_at_calibrated_threshold']
    })
rows.sort(key=lambda row: row['experiment'])
out = root.parent / 'multiseed_summary.csv'
with out.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]) if rows else ['experiment'])
    writer.writeheader(); writer.writerows(rows)
print('已完成实验数：', len(rows))
for row in rows: print(row)
print('汇总 CSV：', out)


已完成实验数： 0
汇总 CSV： runs/internal/34_tsinghua_ev_brand2_four_models_three_seeds_folds1to4/multiseed_summary.csv
